# Notebook 3: 加权融合心率估计

核心实验: 用窗口级分布加权融合多个简单场景参数的心率估计。

对波比跳数据, 分别用各简单运动的最优参数求解, 然后按分类器输出的分布加权。

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

PROJECT_ROOT = Path(r"D:\data\PPG_HeartRate\Algorithm\outline-PPGtoHR")
DATA_DIR = PROJECT_ROOT / "20260418test_python"
ARTIFACTS_DIR = PROJECT_ROOT / "docs" / "research" / "artifacts"
sys.path.insert(0, str(PROJECT_ROOT / "python" / "src"))

from ppg_hr.params import SolverParams
from ppg_hr.core.heart_rate_solver import solve_from_arrays, load_raw_data

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

## 1. 加载前置数据

In [ ]:
with open(ARTIFACTS_DIR / "simple_params.pkl", "rb") as f:
    params_data = pickle.load(f)

simple_params = params_data["simple_params"]
burpee_baseline = params_data["burpee_baseline"]

print("简单运动参数集:")
for name, p in simple_params.items():
    print(f"  {name}: fs={p.fs_target}, order={p.max_order}, smooth={p.smooth_win_len}")

In [ ]:
burpee_proba = np.load(ARTIFACTS_DIR / "burpee_window_distributions.npy")
with open(ARTIFACTS_DIR / "burpee_meta.pkl", "rb") as f:
    burpee_meta = pickle.load(f)

print(f"波比跳概率分布: {burpee_proba.shape}")
print(f"类别: {burpee_meta['class_names']}")

## 2. 选择波比跳测试文件并加载数据

In [ ]:
BURPEE_DIR = DATA_DIR / "bobi"
burpee_csvs = sorted(BURPEE_DIR.glob("multi_bobi*.csv"))
burpee_csvs = [f for f in burpee_csvs if not f.stem.endswith("_ref")]

print("可用波比跳文件:")
for f in burpee_csvs:
    print(f"  {f.name}")

BURPEE_CSV = burpee_csvs[0]
BURPEE_REF = BURPEE_CSV.with_name(BURPEE_CSV.stem + "_ref" + BURPEE_CSV.suffix)
print(f"\n使用: {BURPEE_CSV.name}")

In [ ]:
dummy_params = SolverParams(file_name=str(BURPEE_CSV), ref_file=str(BURPEE_REF))
raw_data, ref_data = load_raw_data(dummy_params)

print(f"原始数据: {raw_data.shape}")
print(f"参考数据: {ref_data.shape}")

## 3. 用各简单运动参数分别求解

In [ ]:
class_names = burpee_meta["class_names"]
print(f"分类器类别顺序: {class_names}")

for cn in class_names:
    if cn not in simple_params:
        print(f"警告: '{cn}' 缺少对应参数集!")

In [ ]:
results = {}

for class_name in class_names:
    if class_name not in simple_params:
        continue

    params_i = simple_params[class_name].replace(
        file_name=str(BURPEE_CSV), ref_file=str(BURPEE_REF),
    )

    print(f"求解 {class_name} ...", end=" ")
    result_i = solve_from_arrays(raw_data, ref_data, params_i)

    results[class_name] = {
        "result": result_i,
        "time": result_i.HR[:, 0],
        "ref_hr_hz": result_i.HR[:, 1],
        "fus_hf_hz": result_i.HR[:, 5],
        "fus_acc_hz": result_i.HR[:, 6],
        "motion_flag": result_i.HR[:, 7],
        "n_windows": result_i.HR.shape[0],
    }
    print(f"{result_i.HR.shape[0]} 窗口, AAE(HF)={result_i.err_fus_hf:.2f} BPM")

## 4. 时间轴对齐

In [ ]:
all_times = [r["time"] for r in results.values()]
t_min = max(t[0] for t in all_times)
t_max = min(t[-1] for t in all_times)
t_common = np.arange(t_min, t_max + 1, 1.0)

print(f"公共时间网格: [{t_min:.1f}, {t_max:.1f}]s, {len(t_common)} 点")

aligned_hr = {}
aligned_ref = None
aligned_motion = None

for cn, r in results.items():
    f_interp = interp1d(r["time"], r["fus_hf_hz"], kind="linear", fill_value="extrapolate")
    aligned_hr[cn] = f_interp(t_common)

    if aligned_ref is None:
        f_ref = interp1d(r["time"], r["ref_hr_hz"], kind="linear", fill_value="extrapolate")
        aligned_ref = f_ref(t_common)
        f_motion = interp1d(r["time"], r["motion_flag"], kind="nearest", fill_value="extrapolate")
        aligned_motion = f_motion(t_common)

In [ ]:
# 对齐概率分布
burpee_times = burpee_meta["times"]
file_mask = np.ones(len(burpee_times), dtype=bool)
proba = burpee_proba[file_mask]
bt = burpee_times[file_mask]

aligned_proba = np.zeros((len(t_common), len(class_names)))
for i in range(len(class_names)):
    f_prob = interp1d(bt, proba[:, i], kind="linear", fill_value="extrapolate")
    aligned_proba[:, i] = f_prob(t_common)

aligned_proba = np.clip(aligned_proba, 0, None)
row_sums = aligned_proba.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
aligned_proba = aligned_proba / row_sums
print(f"对齐概率分布: {aligned_proba.shape}")

## 5. 加权融合

In [ ]:
hr_matrix = np.zeros((len(class_names), len(t_common)))
for i, cn in enumerate(class_names):
    hr_matrix[i] = aligned_hr.get(cn, aligned_ref)

weighted_hr_hz = np.sum(aligned_proba * hr_matrix, axis=0)
uniform_hr_hz = np.mean(hr_matrix, axis=0)

ref_bpm = aligned_ref * 60
weighted_hr_bpm = weighted_hr_hz * 60
uniform_hr_bpm = uniform_hr_hz * 60
motion_mask = aligned_motion > 0.5
rest_mask = ~motion_mask

## 6. 误差计算

In [ ]:
def compute_aae(pred_bpm, ref_bpm, mask=None):
    if mask is not None:
        pred_bpm = pred_bpm[mask]
        ref_bpm = ref_bpm[mask]
    return np.mean(np.abs(pred_bpm - ref_bpm))


print("=" * 60)
print("AAE (BPM):")
print("=" * 60)

aae_weighted_all = compute_aae(weighted_hr_bpm, ref_bpm)
aae_weighted_rest = compute_aae(weighted_hr_bpm, ref_bpm, rest_mask)
aae_weighted_motion = compute_aae(weighted_hr_bpm, ref_bpm, motion_mask)
print(f"加权融合:   All={aae_weighted_all:.2f}, Rest={aae_weighted_rest:.2f}, Motion={aae_weighted_motion:.2f}")

aae_uniform_all = compute_aae(uniform_hr_bpm, ref_bpm)
aae_uniform_rest = compute_aae(uniform_hr_bpm, ref_bpm, rest_mask)
aae_uniform_motion = compute_aae(uniform_hr_bpm, ref_bpm, motion_mask)
print(f"均匀平均:   All={aae_uniform_all:.2f}, Rest={aae_uniform_rest:.2f}, Motion={aae_uniform_motion:.2f}")

for cn in class_names:
    if cn in results:
        single_bpm = aligned_hr[cn] * 60
        aae = compute_aae(single_bpm, ref_bpm)
        aae_r = compute_aae(single_bpm, ref_bpm, rest_mask)
        aae_m = compute_aae(single_bpm, ref_bpm, motion_mask)
        print(f"{cn:15s}: All={aae:.2f}, Rest={aae_r:.2f}, Motion={aae_m:.2f}")

if burpee_baseline:
    print(f"\n直接优化基线: {burpee_baseline.get('min_err_hf', 0):.2f} BPM")
print("=" * 60)

## 7. 可视化

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

ax = axes[0]
ax.plot(t_common, ref_bpm, "k-", label="参考 HR", linewidth=2)
ax.plot(t_common, weighted_hr_bpm, "r-", label="加权融合", linewidth=1.5, alpha=0.8)
motion_starts = np.where(np.diff(motion_mask.astype(int)) == 1)[0]
motion_ends = np.where(np.diff(motion_mask.astype(int)) == -1)[0]
for s, e in zip(motion_starts, motion_ends):
    ax.axvspan(t_common[s], t_common[min(e, len(t_common)-1)], alpha=0.1, color="orange")
ax.set_ylabel("HR (BPM)")
ax.set_title("加权融合 vs 参考")
ax.legend()

ax = axes[1]
ax.plot(t_common, ref_bpm, "k-", label="参考 HR", linewidth=2)
colors = plt.cm.Set2(np.linspace(0, 1, len(class_names)))
for i, cn in enumerate(class_names):
    if cn in results:
        ax.plot(t_common, aligned_hr[cn] * 60, "--", color=colors[i],
                label=cn.replace("_", " ").title(), linewidth=1, alpha=0.7)
ax.set_ylabel("HR (BPM)")
ax.set_title("各简单运动参数集独立求解")
ax.legend(fontsize=8, ncol=2)

ax = axes[2]
ax.plot(t_common, np.abs(weighted_hr_bpm - ref_bpm), "r-", label="加权融合", linewidth=1)
ax.plot(t_common, np.abs(uniform_hr_bpm - ref_bpm), "b--", label="均匀平均", linewidth=1, alpha=0.7)
ax.set_xlabel("时间 (s)")
ax.set_ylabel("绝对误差 (BPM)")
ax.set_title("逐窗口绝对误差")
ax.legend()

plt.tight_layout()
plt.show()

## 8. 保存结果

In [ ]:
fusion_output = {
    "t_common": t_common,
    "ref_bpm": ref_bpm,
    "weighted_hr_bpm": weighted_hr_bpm,
    "uniform_hr_bpm": uniform_hr_bpm,
    "single_hr_bpm": {cn: aligned_hr[cn] * 60 for cn in class_names if cn in aligned_hr},
    "aligned_proba": aligned_proba,
    "motion_mask": motion_mask,
    "class_names": class_names,
    "aae_weighted": {"all": aae_weighted_all, "rest": aae_weighted_rest, "motion": aae_weighted_motion},
    "aae_uniform": {"all": aae_uniform_all, "rest": aae_uniform_rest, "motion": aae_uniform_motion},
    "burpee_csv": str(BURPEE_CSV),
}

with open(ARTIFACTS_DIR / "fusion_results.pkl", "wb") as f:
    pickle.dump(fusion_output, f)
print(f"已保存: {ARTIFACTS_DIR / 'fusion_results.pkl'}")